In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [3]:
import os
import random
import json
import time
import math
import zipfile
import shutil
from torch.utils.data import DataLoader

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
# You can use any of the 4 train files or all the files for pre-training the legal model
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Training configuration
TARGET_TOKENS = 500_000_000
MAX_SEQ_LEN = 1024  #CONTEXT LENGTH
TRAIN_STEPS = 1
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

N_EMBD = 512
N_LAYER = 8
N_HEAD = 8

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.1
TOP_P = 0.95
VOCAB_SIZE = 8000
MODEL_DIM = 512
MODEL_LAYERS = 10
MODEL_FF = 4096
N_HEADS = 8
N_KV_HEADS = 8
BATCH_SIZE = 16
# Kaggle output
OUTPUT_DIR = "/kaggle/working/cs781_phase1"
MODEL_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "legal-gpt2-final"
)
PREDICTION_PATH = os.path.join(
    OUTPUT_DIR,
    "prediction.json"
)
SUBMISSION_ZIP = os.path.join(
    OUTPUT_DIR,
    "submission.zip"
)
LOSS_PLOT_PATH = os.path.join(
    OUTPUT_DIR,
    "training_validation_loss.png"
)
os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)
print("Configuration loaded.")

cuda
Configuration loaded.


In [4]:

class Node:
    def __init__(self,value, previous, next, index):
        self.value=value
        self.previous=previous
        self.next=next
        self.index=index


class LinkedArray:
    def __init__(self, items):
        self._array=[None for _ in range(len(items))]
        previous=None
        for i in range(len(items)):
            node=Node(items[i], previous, None, i)
            if previous is not None:
                node.previous=previous
                previous.next=node
            self._array[i]=node
            previous=node

    def get_previous_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].previous == None:
            return None
        return self._array[index].previous.index

    def get_next_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None:
            return None
        return self._array[index].next.index

    def get_by_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index] == None:
            return None
        return self._array[index].value

    def get_second_next_index(self, index):
        if self._array[index] == None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None or self._array[index].next.next == None:
            return None
        return self._array[index].next.next.index

    def len(self):
        return len(self._array)
    def exists(self, index):
        return (
            0 <= index < len(self._array)
            and self._array[index] is not None
        )

    def replace_pair(self, index, new_item):
        if index > len(self._array)-2 or self._array[index] == None or self._array[index].next == None:
            raise ValueError("Invalid index ",index,new_item)
        self._array[index].value = new_item
        self._array[self._array[index].next.index] = None
        self._array[index].next = self._array[index].next.next
        if self._array[index].next is not None:
            self._array[index].next.previous = self._array[index]


class MaxPriorityMap:
    def __init__(self, heap_key, map_key):
        self._heap_key = heap_key
        self._map_key = map_key
        self._heap = []
        # Map of keys to indexes in the heap
        self._map = {}

    def get_max(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        return self._heap[0]

    def push(self, item):
        self._heap.append(item)
        map_key = self._map_key(item)
        if map_key in self._map:
            raise ValueError("Map key already exist")
        self._map[self._map_key(item)] = len(self._heap) - 1
        self._heapify_up(len(self._heap) - 1)

    def pop(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        self._swap(0, len(self._heap) - 1)
        item = self._heap.pop()
        map_key = self._map_key(item)
        self._map.pop(map_key)
        self._heapify_down(0)
        return item

    def contains(self, map_key):
        return map_key in self._map

    def len(self):
        if len(self._heap) != len(self._map):
            raise ValueError('Invalid state of the priority map')
        return len(self._heap)

    def delete_by_map_key(self, map_key):
        if map_key not in self._map:
            raise ValueError("Map key does not exist")
        heap_index = self._map[map_key]
        item = self._heap[heap_index]
        self._swap(heap_index, len(self._heap) - 1)
        self._heap.pop()
        self._map.pop(map_key)
        if (heap_index < len(self._heap)):
            self._heapify_down(heap_index)
            self._heapify_up(heap_index)
        return item

    def _heapify_up(self, i):
        while i > 0 and self._heap_key(self._heap[i]) > self._heap_key(self._heap[self._parent(i)]):
            self._swap(i, self._parent(i))
            i = self._parent(i)

    def _heapify_down(self, i):
        largest = i
        left = self._left_child(i)
        right = self._right_child(i)

        if left < len(self._heap) and self._heap_key(self._heap[left]) > self._heap_key(self._heap[largest]):
            largest = left

        if right < len(self._heap) and self._heap_key(self._heap[right]) > self._heap_key(self._heap[largest]):
            largest = right

        if largest != i:
            self._swap(i, largest)
            self._heapify_down(largest)

    def _parent(self, i):
        return (i - 1) // 2

    def _left_child(self, i):
        return 2 * i + 1

    def _right_child(self, i):
        return 2 * i + 2

    def _swap(self, i, j):
        self._map[self._map_key(self._heap[i])] = j
        self._map[self._map_key(self._heap[j])] = i
        self._heap[i], self._heap[j] = self._heap[j], self._heap[i]


class StatsEntry:
    def __init__(self, pair, positions):
        self.pair = pair
        self.positions = positions

class TokenizerTrainer:
        def __init__(self, tokens_list, itos, num_merges):
            self.tokens_list = tokens_list
            self.itos = itos
            self._num_merges = num_merges
            self.stoi={}

        def train(self, next_token):
            self._positions=[LinkedArray(tokens) for tokens in self.tokens_list]
            self.get_stats()
            merges_done=0
            print("requested merges",self._num_merges)
            while  merges_done<self._num_merges:
                if self._stats.len() == 0:
                   print("STOPPED: stats became empty")
                   print("Merges completed:", merges_done)
                   break
                merge_stat = self._stats.pop()
                (current_token , next_token) = self.update_itos_and_stoi(next_token, merge_stat)
                #iterate through posistions replace with merge pair
                for position in list(merge_stat.positions):

                    if position not in merge_stat.positions:
                        continue
                    input_index = position[0]
                    token_index = position[1]
                    # print("current_token",current_token)
                    if not self._positions[input_index].exists(token_index):
                        merge_stat.positions.discard(position)
                        continue
                    self.update_right_token(input_index, token_index, merge_stat, current_token)
                    self.update_left_token(input_index, token_index, merge_stat, current_token)
                    self._positions[input_index].replace_pair(token_index, current_token)
                    merge_stat.positions.remove((input_index, token_index))
                merges_done += 1
            return self.itos


        def update_right_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           next_token_idx = position.get_next_index(token_index)
           right_token_idx = position.get_second_next_index(token_index)
           if right_token_idx == None:
               return
           pair = (merge_stat.pair[1], position.get_by_index(right_token_idx))
           self.remove_pair_from_stats(merge_stat, pair, input_index, next_token_idx)
           new_pair = (current_token, pair[1])
           self.add_position_to_pair(new_pair, input_index, token_index)

        def update_left_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           left_token_idx = position.get_previous_index(token_index)
           if left_token_idx == None:
               return
           pair = ( position.get_by_index(left_token_idx), merge_stat.pair[0])
           self.remove_pair_from_stats(merge_stat, pair, input_index, left_token_idx)
           new_pair = (pair[0] , current_token)
           self.add_position_to_pair(new_pair, input_index, left_token_idx)

        def add_position_to_pair(self, new_pair, input_index, token_index):
            if self._stats.contains(new_pair):
                stat = self._stats.delete_by_map_key(new_pair)
            else:
                stat = StatsEntry(new_pair, set())

            stat.positions.add((input_index, token_index))
            self._stats.push(stat)


        def remove_pair_from_stats(self, merge_stat, pair, input_index, token_index):
           if pair == merge_stat.pair:
               merge_stat.positions.discard((input_index, token_index))
           else:
                if self._stats.contains(pair):
                    stat = self._stats.delete_by_map_key(pair)
                    stat.positions.discard((input_index, token_index))
                    if len(stat.positions) != 0:
                        self._stats.push(stat)


        def update_itos_and_stoi(self, next_token, merge_stat):
           merge_pair = self.itos[merge_stat.pair[0]] + self.itos[merge_stat.pair[1]]
           if merge_pair in self.stoi:
               current_token = self.stoi[merge_pair]
           else:
               current_token = next_token
               self.itos[current_token] = merge_pair
               self.stoi[merge_pair] = current_token
               next_token = current_token+1
           return  (current_token , next_token)

        def get_stats(self):
             self._stats = MaxPriorityMap(
                                heap_key=lambda item: (len(item.positions),tuple(-x for x in item.pair)),
                                map_key=lambda item: item.pair
                            )
             stats={}
             for  text_id in range(len(self.tokens_list)):
                 tokens = self.tokens_list[text_id]
                 for idx in range(len(tokens)-1):
                     pair = (tokens[idx], tokens[idx+1])
                     if pair not in stats:
                         stats[pair]=StatsEntry(pair, set())
                     stats[pair].positions.add((text_id, idx))
             for key in stats.keys():
                self._stats.push(stats[key])

class TokenNode:
    def __init__(self, byte_value, token, parent, children):
        self.parent = parent
        self.byte_value = byte_value
        self.token = token
        self.children = children

class ToTokensConverter:

    def __init__(self, itos):
        self.itos = itos
        self.unk_token=1
        self._init_token_tree()

    def to_tokens(self, strings):
        return [
            self._to_tokens(string)
            for string in strings
        ]

    def _to_tokens(self, string):

        words = string.split()
        tokens = []
        for word in words:
            data = word.encode("utf-8") + b"</w>"
            word_tokens = self._tokenize_word(data)
            tokens.extend(word_tokens)
        return tokens

    def _tokenize_word(self, data):

        tokens = []

        byte_index = 0

        while byte_index < len(data):

            current_nodes = self._token_roots

            last_node_with_token = None
            last_index_with_token = byte_index

            while byte_index < len(data):

                byte_value = data[byte_index]

                if byte_value in current_nodes:

                    node = current_nodes[byte_value]

                    if node.token is not None:
                        last_node_with_token = node
                        last_index_with_token = byte_index

                    current_nodes = node.children
                    byte_index += 1

                else:
                    break

            if last_node_with_token is None:
                tokens.append(self.unk_token)

                # Move forward by one byte
                byte_index += 1

            else:
                tokens.append(last_node_with_token.token)
                byte_index = last_index_with_token + 1

        return tokens

    def _init_token_tree(self):

        self._token_roots = {}

        for token, token_bytes in self.itos.items():

            if token < 4:
                continue

            current_nodes = self._token_roots
            last_node = None

            for byte_value in token_bytes:

                if byte_value in current_nodes:

                    last_node = current_nodes[byte_value]

                else:

                    last_node = TokenNode(
                        byte_value,
                        None,
                        last_node,
                        {}
                    )

                    current_nodes[byte_value] = last_node

                current_nodes = last_node.children

            if last_node.token is not None:
                raise KeyError("The token is already taken")

            last_node.token = token

class BPETokenizer:
    def __init__(self, num_merges):
        self._num_merges = num_merges
        self.itos = {}
    def get_vocab_size(self):
        return len(self.itos)
    def train(self, texts):
        self.create_base_vocab()

        tokens_list = self.get_ids_from_text(texts)
        # for tokens in tokens_list:
        #     print(tokens)
        trainer = TokenizerTrainer(
                        tokens_list,
                        self.itos,
                        self._num_merges
                    )

        trainer.train(261)

        return self.itos, tokens_list

    def encode(self, text):
        converter = ToTokensConverter(self.itos)
        return converter.to_tokens([text])[0]

    def decode(self, tokens):
        return (
                b"".join(
                    self.itos[token]
                    for token in tokens
                )
                .decode("utf-8", errors="replace")
                .replace("</w>", " ")
                .replace("<s>", "")
                .replace("</s>", "")
                .replace("<pad>", "")
                .strip()
            )

    def create_base_vocab(self):
        self.itos = {
            0: b"<pad>",
            1: b"<unk>",
            2: b"<s>",
            3: b"</s>",
            260 : b"</w>"
        }
        for i in range(256):
            self.itos[i + 4] = bytes([i])

    def get_ids_from_text(self, texts):
        tokens_list=[]
        for text in texts:
            words = text.split()
            for word in words:
                word_bytes = [byte + 4 for byte in word.encode("utf-8")]
                word_bytes.append(260)
                tokens_list.append(word_bytes)
        return tokens_list

In [5]:
import numpy as np
import pickle

# Paths
bpe_path = "/kaggle/input/datasets/ksaiveerendraprasad/tokenizer-results/project_bpe_16k.pkl"
train_path = "/kaggle/input/datasets/ksaiveerendraprasad/mha-kernel-results/train_tokens2.npy"
val_path = "/kaggle/input/datasets/ksaiveerendraprasad/tokenizer-results/val_tokens1 (1).npy"
train2_path = "/kaggle/input/datasets/ksaiveerendraprasad/tokenizer-results/train_tokens1 (2).npy"

# Load BPE tokenizer
with open(bpe_path, "rb") as f:
    bpe = pickle.load(f)

# Load token arrays
train_tokens = np.load(train_path, mmap_mode="r")
val_tokens = np.load(val_path, mmap_mode="r")
train2_tokens = np.load(train2_path, mmap_mode = "r")

print("BPE vocab size:", len(bpe.itos))
print("Train tokens:", len(train_tokens))
print("Validation tokens:", len(val_tokens))
print("Train 2 tokens" , len(train2_tokens))

BPE vocab size: 7999
Train tokens: 200000000
Validation tokens: 5000000
Train 2 tokens 199999998


In [6]:
print(np.array_equal(train_tokens, train2_tokens))

False


In [7]:
itos = bpe.itos.copy()

MASK_ID = max(itos.keys()) + 1
itos[MASK_ID] = b"[MASK]"

bpe.itos = itos
bpe.stoi = {token: idx for idx, token in itos.items()}
IGNORE = -1

In [8]:
from torch.utils.data import Dataset, DataLoader
from typing import List
class TokenDataset(Dataset):
    def __init__(self, token_ids: List[int], seq_len: int):
        self.ids = torch.tensor(token_ids, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        return max(0, (len(self.ids) - 1) // self.seq_len)

    def __getitem__(self, i):
        start = i * self.seq_len
        x = self.ids[start : start + self.seq_len]
        return x

In [9]:
train_dataset = TokenDataset(train_tokens, MAX_SEQ_LEN)
val_dataset = TokenDataset(val_tokens, MAX_SEQ_LEN)

In [10]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=True
)

In [11]:
def build_hybrid_batch(ids, mask_id, causal_fraction=0.5, mask_prob=0.15):
    B, T = ids.shape
    input_ids = ids.clone()

    targets = torch.full_like(ids, IGNORE)
    is_causal = (torch.rand(B, device=ids.device) < causal_fraction)
    
    causal_rows = is_causal
    targets[causal_rows, :-1] = ids[
        causal_rows, 1:
    ]

    # --------------------------------------------------
    # Masked examples
    # --------------------------------------------------

    masked_rows = ~is_causal

    mask = (torch.rand_like( ids.float()) < mask_prob)

    mask[:, 0] = False

    # Only mask masked-mode examples
    mask &= masked_rows[:, None]

    # Original tokens become targets
    targets[mask] = ids[mask]

    # Replace input tokens with [MASK]
    input_ids[mask] = mask_id

    return {
        "input_ids": input_ids,
        "targets": targets,
        "is_causal": is_causal
    }

In [12]:

"""Rotary Positional Embeddings, implemented from scratch."""
import torch


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """
    Returns (cos, sin) tensors of shape (max_seq_len, d_head) ready to be applied to
    query/key tensors of head dimension d_head. d_head must be even. This scaffold uses
    the split-half rotate_half convention: channel i is paired with channel i+d_head/2.
    """
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    inverse_frequencies =torch.tensor([1/base ** (2*i/d_head) for i in range(d_head//2)],dtype = torch.float32)
    positions = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(positions, inverse_frequencies)
    angles = torch.cat([freqs, freqs], dim=-1)
    return angles.cos(), angles.sin()



def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree
    rotation step used by the "rotate_half" formulation of RoPE."""
    # TODO
    d_head = x.shape[-1]
    x1, x2 = x[...,:d_head//2], x[...,d_head//2:]
    return torch.cat([-x2, x1], dim=-1)



def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (B, n_heads, T, d_head) -- a query or key tensor.
    cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T.
    Returns the rotated tensor, same shape as x.
    Formula: x_rotated = x * cos + rotate_half(x) * sin
    """
    # TODO: broadcast cos/sin to (1, 1, T, d_head) and apply the formula above
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    x_rotated = x*cos + rotate_half(x)*sin
    return x_rotated



In [13]:

# TODO 3.2: integrate RoPE through attention and the decoder architecture in this section.
"""Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)


        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x: torch.Tensor,attn_mask=None) -> torch.Tensor:
        B, T, C = x.shape
        q = self.q_proj(x).reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).reshape(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).reshape(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q, self.rope_cos[:T], self.rope_sin[:T])
            k = apply_rope(k, self.rope_cos[:T], self.rope_sin[:T])
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim = 1)
            v = v.repeat_interleave(self.n_rep, dim = 1)
       
        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=self.attn_dropout.p if self.training else 0.0
        )
        out = out.transpose(1, 2).reshape(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out



In [14]:

import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F



@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook



def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    PE = torch.zeros(max_len, d_model)
    for pos in range(max_len):
      for i in range(d_model//2):
        PE[pos, 2*i] = math.sin(pos / 10000**(2*i/d_model))
        PE[pos, 2*i+1] = math.cos(pos / 10000**(2*i/d_model))
    return PE


def build_attn_mask(is_causal, padding_mask, T):
    """is_causal (B,) bool; padding_mask (B,T) bool, True = padding, or None -> (B,1,T,T) bool."""
    full = torch.ones(T, T, dtype=torch.bool, device=is_causal.device)
    allow = torch.where(is_causal[:, None, None, None], torch.tril(full), full)
    if padding_mask is not None:
        allow = allow & ~padding_mask[:, None, None, :].to(torch.bool)
    return allow
    
class EncoderFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )   # unchanged from Question 1

    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = MultiHeadAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = EncoderFeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x, attn_mask=None):
        x = x + self.attn(self.ln1(x), attn_mask)
        x = x + self.ffn(self.ln2(x))
        return x   # unchanged from Question 1


class EncoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(
            cfg.d_model,
            cfg.vocab_size,
            bias=False
        )
        
        self.lm_head.weight = self.tok_emb.weight
                

    def forward(self, idx, padding_mask=None, is_causal=None):                 # CHANGED: is_causal
            B, T = idx.shape
            if padding_mask is not None:
                if padding_mask.shape != (B, T):
                    raise ValueError("padding_mask must have shape (B, T)")
                padding_mask = padding_mask.to(device=idx.device, dtype=torch.bool)
            if is_causal is None:                                                  # old behaviour: bidirectional
                is_causal = torch.zeros(B, dtype=torch.bool, device=idx.device)
            attn_mask = build_attn_mask(is_causal, padding_mask, T)
            x = self.tok_emb(idx)
            if self.cfg.pos_emb_type == "sinusoidal":
                x = x + self.pos_emb[:T].unsqueeze(0)
            for block in self.blocks:
                x = block(x, attn_mask=attn_mask)
            x = self.ln_f(x)
            return self.lm_head(x) 


In [15]:
@torch.no_grad()
def evaluate_model(
    model,
    val_batches,
    device=device
):
    model.eval()

    losses = []

    for batch in val_batches:

        input_ids = batch["input_ids"].to(
            device, dtype=torch.long, non_blocking=True
        )

        targets = batch["targets"].to(
            device, dtype=torch.long, non_blocking=True
        )

        is_causal = batch["is_causal"].to(
            device, non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = model(
                input_ids,
                is_causal=is_causal
            )

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
                ignore_index=IGNORE
            )

        losses.append(loss.item())

    return sum(losses) / len(losses)

In [16]:
def train_model(
    model,
    train_batch,
    val_batch,
    optimizer,
    steps=1000,
    mask_id=MASK_ID,
    causal_fraction=0.5,
    mask_prob=0.15,
    eval_every=100,
    device=device,
    save_path="hybrid_encoder.pt"
):
    model.to(device)
    history = []
    best_state = None
    best_ppl  = float("inf")

    scaler = torch.amp.GradScaler("cuda")
    start = time.time()
    for step in range(1, steps + 1):

        model.train()

        # ------------------------------------------------
        # Get precomputed hybrid batch
        # ------------------------------------------------

        batch = train_batch[(step - 1) % len(train_batch)]

        input_ids = batch["input_ids"].to(
            device, dtype=torch.long
        )

        targets = batch["targets"].to(
            device, dtype=torch.long
        )

        is_causal = batch["is_causal"].to(
            device
        )

        # ------------------------------------------------
        # Forward
        # ------------------------------------------------

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = model(
                input_ids,
                is_causal=is_causal
            )

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
                ignore_index=IGNORE
            )

        # ------------------------------------------------
        # Backprop
        # ------------------------------------------------

        optimizer.zero_grad(set_to_none=True)

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        # ------------------------------------------------
        # Validation
        # ------------------------------------------------
        if step % 100 == 0 and step % eval_every != 0:
            print(
                f"Step {step:4d} | "
                f"Train Loss: {loss.item():.4f} | "
                f"Time : {(time.time()-start)/60:.2f} mins"
            )
        if step == 1 or step % eval_every == 0:

            val_loss = evaluate_model(
                model,
                val_batch
            )
            val_ppl = np.exp(val_loss)

            print(
                f"Step {step:4d} |"
                f"Train Loss: {loss.item():.4f} | "
                f"Val PPL: {val_ppl:.2f} | "
                f"Time : {(time.time()-start)/60:.2f} mins"
            )
            history.append({
                "step": step,
                "train_loss": loss.item(),
                "val_ppl": val_ppl,
                "elapsed_time":(time.time()-start)/60
            })

            if val_ppl < best_ppl:
                best_ppl = val_ppl
                best_state = {
                        k: v.detach().cpu().clone()
                        for k, v in model.state_dict().items()
                    }
                torch.save({
                    "step": step,
                    "model_state_dict": best_state,
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_ppl": best_ppl
                }, save_path)

    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(device)
    return history, best_ppl

In [17]:
cfg = LMConfig(
    vocab_size=VOCAB_SIZE,
    d_model=MODEL_DIM,
    n_layers=MODEL_LAYERS,
    n_heads=N_HEADS,
    n_kv_heads=N_KV_HEADS,
    d_ff=MODEL_FF,
    max_seq_len=MAX_SEQ_LEN,
    dropout=0.1,
    pos_emb_type="rope",
)

model = EncoderLM(cfg)

if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    model = torch.nn.DataParallel(model)

model = model.to(device)
optimizer= torch.optim.AdamW(model.parameters(), lr =3e-4, weight_decay = 0.1)

Using 2 GPUs


In [18]:
def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())
print("total parameters : ",count_parameters(model))

total parameters :  56592384


In [19]:
def precompute_hybrid_batches(
    loader,
    mask_id,
    causal_fraction=0.5,
    mask_prob=0.15
):
    hybrid_batches = []

    for ids in loader:
        ids = ids.to(dtype=torch.long)

        batch = build_hybrid_batch(
            ids,
            mask_id=mask_id,
            causal_fraction=causal_fraction,
            mask_prob=mask_prob
        )

        hybrid_batches.append({
            "input_ids": batch["input_ids"],
            "targets": batch["targets"],
            "is_causal": batch["is_causal"]
        })

    return hybrid_batches

In [20]:
train_batch = precompute_hybrid_batches(train_loader, MASK_ID)
val_batch  = precompute_hybrid_batches(val_loader, MASK_ID)
del train_loader
del val_loader

# Optional: free Python memory
import gc
gc.collect()

30

In [21]:

checkpoint = torch.load("/kaggle/input/datasets/ksaiveerendraprasad/hybridencoder-model/hybrid_encoder.pt", map_location=device, weights_only=False)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

# step = checkpoint["step"]
# best_ppl = checkpoint["best_ppl"]

model.to(device)

print("Loaded successfully!")
# print("Step:", step)
# print("Best PPL:", best_ppl)

Loaded successfully!


In [22]:
history = train_model(
    model=model,
    train_batch=train_batch,
    val_batch=val_batch,
    optimizer=optimizer,
    steps=12500,
    mask_id=MASK_ID,
    causal_fraction=0.5,
    mask_prob=0.15,
    eval_every=100
)

Step    1 |Train Loss: 2.3771 | Val PPL: 4.91 | Time : 0.74 mins
Step  100 |Train Loss: 2.3652 | Val PPL: 5.00 | Time : 2.17 mins
Step  200 |Train Loss: 2.5210 | Val PPL: 5.01 | Time : 3.61 mins
Step  300 |Train Loss: 2.3538 | Val PPL: 5.03 | Time : 5.04 mins
Step  400 |Train Loss: 2.2865 | Val PPL: 5.04 | Time : 6.48 mins
Step  500 |Train Loss: 2.4865 | Val PPL: 5.04 | Time : 7.91 mins
Step  600 |Train Loss: 2.2594 | Val PPL: 5.03 | Time : 9.35 mins
Step  700 |Train Loss: 2.1970 | Val PPL: 5.07 | Time : 10.78 mins
Step  800 |Train Loss: 2.3382 | Val PPL: 5.04 | Time : 12.21 mins
Step  900 |Train Loss: 2.2171 | Val PPL: 5.03 | Time : 13.64 mins
Step 1000 |Train Loss: 2.5475 | Val PPL: 5.03 | Time : 15.08 mins
Step 1100 |Train Loss: 2.3101 | Val PPL: 5.01 | Time : 16.51 mins
Step 1200 |Train Loss: 2.0583 | Val PPL: 5.01 | Time : 17.94 mins
Step 1300 |Train Loss: 2.2582 | Val PPL: 5.03 | Time : 19.38 mins
Step 1400 |Train Loss: 1.9245 | Val PPL: 4.98 | Time : 20.81 mins
Step 1500 |Train 

In [23]:
import torch


@torch.no_grad()
def generate_text(
    model,
    tokenizer,
    prompt,
    max_words=128,
    temperature=0.8,
    top_k=50,
    device=device
):
    model.eval()

    # Handle DataParallel
    cfg = (
        model.module.cfg
        if isinstance(model, torch.nn.DataParallel)
        else model.cfg
    )

    # --------------------------------------------------
    # 1. Tokenize prompt
    # --------------------------------------------------
    input_ids = tokenizer.encode(prompt)

    input_ids = torch.tensor(
        input_ids,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)
    generated_tokens = []
    # --------------------------------------------------
    # 2. Generate one token at a time
    # --------------------------------------------------
    word_count = 0
    while word_count < max_words:

        # Keep maximum context length
        idx = input_ids[:, -cfg.max_seq_len:]

        # Always causal during generation
        is_causal = torch.ones(
            idx.size(0),
            dtype=torch.bool,
            device=device
        )

        # Forward
        logits = model(
            idx,
            is_causal=is_causal
        )

        # Last position
        logits = logits[:, -1, :]

        # Temperature
        logits = logits / temperature

        # Top-k
        if top_k is not None:
            k = min(top_k, logits.size(-1))

            values, indices = torch.topk(
                logits,
                k
            )

            filtered_logits = torch.full_like(
                logits,
                float("-inf")
            )

            filtered_logits.scatter_(
                1,
                indices,
                values
            )

            logits = filtered_logits

        # Probabilities
        probs = torch.softmax(
            logits,
            dim=-1
        )

        # Sample
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )
        token_id = next_token.item()

        token_bytes = bpe.itos[token_id]

        if token_bytes.endswith(b"</w>"):
            word_count += 1
        generated_tokens.append(token_id)
        # Append
        input_ids = torch.cat(
            [input_ids, next_token],
            dim=1
        )
        

    # --------------------------------------------------
    # 3. Decode
    # --------------------------------------------------
    return tokenizer.decode(
       generated_tokens
    )

In [24]:
from huggingface_hub import hf_hub_download

test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file:")
print(test_file_path)
with open(test_file_path,"r", encoding="utf-8") as f:
    test_data = json.load(f)

test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

Test file:
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json


In [25]:
import time
import signal

predictions = {}
failed_predictions = {}

TIMEOUT_SECONDS = 120
MAX_RETRIES = 3

def timeout_handler(signum, frame):
    raise TimeoutError("Generation exceeded 120 seconds")

signal.signal(signal.SIGALRM, timeout_handler)

for sample_id, data in test_data.items():
    text = data["text"]
    input_ids = bpe.encode(text)

    print(f"\nID: {sample_id}")
    print(f"BPE length: {len(input_ids)}")

    if len(input_ids) == 0:
        print("EMPTY INPUT:", sample_id)
        print("Text:", repr(text[:200]))
        failed_predictions[sample_id] = "Empty input"
        continue

    success = False

    for attempt in range(1, MAX_RETRIES + 1):
        start_time = time.monotonic()

        try:
            print(f"Attempt {attempt}/{MAX_RETRIES}")

            signal.alarm(TIMEOUT_SECONDS)
            try:
                prediction = generate_text(model.module, bpe, text)
            finally:
                signal.alarm(0)

            elapsed = time.monotonic() - start_time

            prediction = (
                prediction.replace("<s>", "")
                .replace("</s>", "")
                .replace("<pad>", "")
            )

            predictions[sample_id] = prediction
            print(f"Success in {elapsed:.1f}s")
            success = True
            break

        except TimeoutError:
            elapsed = time.monotonic() - start_time
            print(f"Timed out after {elapsed:.1f}s")

        except Exception as e:
            elapsed = time.monotonic() - start_time
            print(f"Attempt failed after {elapsed:.1f}s: {e}")

        finally:
            signal.alarm(0)

        if attempt < MAX_RETRIES:
            print("Retrying in 2 seconds...")
            time.sleep(2)

    if not success:
        failed_predictions[sample_id] = "Failed after 3 attempts"
        print(f"Skipping ID {sample_id}")

print("\nCompleted.")
print("Successful predictions:", len(predictions))
print("Failed predictions:", len(failed_predictions))


ID: 1
BPE length: 674
Attempt 1/3
Success in 6.0s

ID: 2
BPE length: 666
Attempt 1/3
Success in 5.7s

ID: 3
BPE length: 586
Attempt 1/3
Success in 4.9s

ID: 4
BPE length: 637
Attempt 1/3
Success in 5.2s

ID: 5
BPE length: 612
Attempt 1/3
Success in 5.4s

ID: 6
BPE length: 619
Attempt 1/3
Success in 4.7s

ID: 7
BPE length: 661
Attempt 1/3
Success in 6.2s

ID: 8
BPE length: 722
Attempt 1/3
Success in 5.4s

ID: 9
BPE length: 646
Attempt 1/3
Success in 5.2s

ID: 10
BPE length: 612
Attempt 1/3
Success in 8.3s

ID: 11
BPE length: 771
Attempt 1/3
Success in 7.7s

ID: 12
BPE length: 897
Attempt 1/3
Success in 6.5s

ID: 13
BPE length: 610
Attempt 1/3
Success in 5.0s

ID: 14
BPE length: 670
Attempt 1/3
Success in 5.0s

ID: 15
BPE length: 671
Attempt 1/3
Success in 5.7s

ID: 16
BPE length: 675
Attempt 1/3
Success in 5.3s

ID: 17
BPE length: 645
Attempt 1/3
Success in 6.1s

ID: 18
BPE length: 710
Attempt 1/3
Success in 6.2s

ID: 19
BPE length: 782
Attempt 1/3
Success in 6.9s

ID: 20
BPE length: 6

In [26]:
import json

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(
        {
            id: {"text": text}
            for id, text in predictions.items()
        },
        f,
        ensure_ascii=False,
        indent=2
    )

In [27]:
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print("Codabench submission created:")
print(SUBMISSION_ZIP)

Codabench submission created:
/kaggle/working/cs781_phase1/submission.zip
